In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps
import datetime
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

/var/folders/fb/2nwyf_mx609gfjxv_jjnb76h0000gn/T/ipykernel_7462/1857937564.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [4]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
         host="processing-2.processing.data.production.internal",
         
#         host="presto.processing.yoda.run",
        # host="processing.processing.data.production.internal",


        port=80,
        protocol='http',
        catalog ='hive',
        username='sameer.kumar@rapido.bike')
    

In [54]:
q="""
   with callingData as (
       select yyyymmdd,count(*) as totalCalls
       from clevertap.captain_callback_galaxy_immutable
       where  yyyymmdd between '20250206' and '20250224'
       and lower(json_extract_scalar(event_props,'$.Service'))='link'
       and json_extract_scalar(event_props,'$.call_type')='IB'
       group by 1
   ),
   orderCount as(
     select yyyymmdd,count(distinct order_id) as totalOrders
     from orders.order_logs_fact
     where yyyymmdd between '20250206' and '20250224'
     and lower(order_status)='dropped'
     and service_level='link'
     group by 1
   )
   select a.yyyymmdd,a.totalCalls,b.totalOrders,(a.totalCalls*1.0/b.totalOrders) as cr
   from callingData a
   left join ordercount b
   on a.yyyymmdd=b.yyyymmdd
"""
postcr=pd.read_sql(q,metabase_connection)

In [9]:
cr['cr']=cr.totalCalls/cr.totalOrders

In [10]:
cr.cr=cr.cr*100

In [23]:
precr['cr']=precr.totalCalls*100/precr.totalOrders

In [24]:
precr[['totalCalls','totalOrders','cr']].to_clipboard()

In [30]:
precr.to_clipboard()

In [34]:
precr

,totalCalls,totalOrders,cr
0,651005,18119317,0.0


In [12]:
q="""
   with callingData as (
       select count(*) as totalCalls
       from clevertap.captain_callback_galaxy_immutable
       where  yyyymmdd between '20250118' and '20250205'
       and lower(json_extract_scalar(event_props,'$.Service'))='link'
       and json_extract_scalar(event_props,'$.call_type')='IB'
   ),
   orderCount as(
     select count(distinct order_id) as totalOrders
     from orders.order_logs_fact
     where yyyymmdd between '20250118' and '20250205'
     and lower(order_status)='dropped'
     and service_level='link'
   )
   select a.totalCalls,b.totalOrders,(a.totalCalls*1.0/b.totalOrders) as cr
   from callingData a
   cross join ordercount b

"""
preCr=pd.read_sql(q,metabase_connection)

In [37]:
precr.to_clipboard()

In [38]:
postCr.to_clipboard()

In [13]:
postCr['cr']=postCr['totalCalls']/postCr['totalOrders']

NameError: name 'postCr' is not defined

In [14]:
preCr

,totalCalls,totalOrders,cr
0,632431,16930175,0.0


In [59]:
preCr.sort_values(by=['yyyymmdd']).to_clipboard()

In [60]:
postcr.sort_values(by=['yyyymmdd']).to_clipboard()

In [ ]:
q="""
with upsdate_date as (
     select userid as captain_id,
     updated_yyyymmdd as updatedate,
     yyyymmdd as createdon,
     yyyymmdd as post_start_date,
     date_format(date_add('day',28,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as post_end_date,
     date_format(date_add('day',-1,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_end_date,
     date_format(date_add('day',-29,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_start_date,
     row_number() over(partition by userid order by updated_yyyymmdd desc) as rank,status
     from canonical.iceberg_domain_payments_captain_autopay_immutable
     where updated_yyyymmdd between'20241001' and '20250126'
     
   ),active_mandate as (
     select *
     from upsdate_date a
     where a.rank=1
     and lower(a.status) like '%mandateactive%'
   ),
   pc as (
      select captain_id,
           case
               when lower(consistency_segment)='inter' then 'Intra'
               when lower(consistency_segment)='intra' then 'Inter'
               else consistency_segment
               end as consistency_segment,performance_segment
    from datasets_internal.captain_subservice_segments_v1 
    where run_date='20250225'
   ),
   appo as(
    select captain_id,gig_segment
    from reports_internal.captain_gig_appography_v4
    where run_date='20250107'
   ),
   am as (
     select a.*,b.consistency_segment,b.performance_segment,c.gig_segment
     from active_mandate a
     left join pc b
     on a.captain_id=b.captain_id
     left join appo c
     on a.captain_id=c.captain_id
   ),
   orders as(
    select yyyymmdd,captain_id,count(distinct order_id) as numOrders
    from orders.order_logs_fact
    where order_status='dropped'
    and yyyymmdd between '20240901' and '2025024'
    and captain_id in (select captain_id from active_mandate)
    group by 1,2
  ),
cm as (
  select yyyymmdd,captain_id,sum(total_login_hours) as login_hrs
  from datasets.captain_metrics as cm
  where yyyymmdd between '20240901' and '20250319'
  and captain_id in (select captain_id from active_mandate)
  group by 1,2
),
totalEarning as( select yyyymmdd,rider_id as captain_id,
    sum(captain_earnings) as earning
    from captain.captain_transactions_snapshot
            where 1=1
            and (transaction_type = 'orders' or transaction_type='incentives')
            and yyyymmdd between '20240901' and '20250224'
            and rider_id in (select captain_id from active_mandate)
    group by 1,2
),
metric_orders as (
   select a.yyyymmdd,a.captain_id,a.numOrders,b.login_hrs
   from orders a
   left join cm b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
   where a.numOrders>0
),
metric_order_earning as(
  select a.yyyymmdd,a.captain_id,a.numOrders,a.login_hrs,b.earning
   from metric_orders a
   left join totalEarning b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
),
 finalTbl as (
    select
         a.captain_id,a.consistency_segment,a.performance_segment,a.gig_segment,
         count(distinct case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.yyyymmdd end) as pre_netDays,
         sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.numOrders else 0 end) as pre_numOrders,
         sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.login_hrs else 0 end) as pre_login_hrs,
         sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.earning else 0 end) as pre_total_earnings,
         count(distinct  case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.yyyymmdd end) as post_netDays,
         sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.numOrders else 0 end) as post_numOrders,
         sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.login_hrs else 0 end) as post_login_hrs,
         sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.earning else 0 end) as post_total_earnings
     from am a
     inner join metric_order_earning b
     on a.captain_id=b.captain_id
     group by 1,2,3,4
 ),
 analysisTbl as (
     select performance_segment,consistency_segment,gig_segment, count(captain_id) as totalCaptains,sum(pre_numOrders) as preOrders,sum(pre_netDays) as preNetDays,sum(pre_login_hrs) as preLoginHrs,
     sum(pre_total_earnings) as preTotalEarnings,
     sum(post_total_earnings) as postTotalEarnings,
     sum(post_numOrders) as postOrders,sum(post_netDays) as postNetDays,
     sum(post_login_hrs) as postLoginHrs
     from finalTbl
     group by 1,2,3
 )
 select * , (RPR_Post - RPR_Pre) as RPR_Change, (Net_days_Post - Net_days_Pre) as Net_days_Change, (LH_Post - LH_Pre) as LH_Change,(Earnings_post - Earnings_pre) as Earnings_change
from (
select performance_segment,consistency_segment,gig_segment,
totalCaptains as Total_Captains,
(preOrders*1.0/totalCaptains) as RPR_Pre,  (preNetDays*1.0/totalCaptains) as Net_days_Pre, (preLoginHrs*1.0 / totalCaptains) as LH_Pre,(preTotalEarnings*1.0/totalCaptains) as Earnings_pre,
(postOrders*1.0/totalCaptains) as RPR_Post,  (postNetDays*1.0/totalCaptains) as Net_days_Post, (postLoginHrs*1.0 / totalCaptains) as LH_Post,(postTotalEarnings*1.0/totalCaptains) as Earnings_post
from analysisTbl
)
"""
GPCautoPay=pd.read_sql(q,metabase_connection)

In [6]:
GPCautoPay

,performance_segment,consistency_segment,gig_segment,Total_Captains,RPR_Pre,Net_days_Pre,LH_Pre,Earnings_pre,RPR_Post,Net_days_Post,LH_Post,Earnings_post,RPR_Change,Net_days_Change,LH_Change,Earnings_change
0,None,None,11_Full_Gig,90,55.8,8.8,72.860164,6927.326192,25.7,4.7,35.359250,3070.472873,-30.1,-4.1,-37.500914,-3856.853319
1,MP,Daily,10_RHA_gig,3711,266.4,21.7,386.044380,34887.605192,329.3,25.3,452.734765,43288.792439,62.9,3.6,66.690385,8401.187247
2,UHP,Sporadic,None,1,604.0,20.0,261.031667,50360.000000,1040.0,20.0,395.201111,81911.220000,436.0,0.0,134.169444,31551.220000
3,None,None,00_non_Gig,89,37.2,5.5,36.477388,4438.898876,17.1,2.9,17.897169,1887.271910,-20.1,-2.6,-18.580218,-2551.626966
4,HP,Intra,01_nonRHA_gig,3,290.7,14.0,255.491667,31368.666667,503.3,20.7,389.870926,53772.666667,212.6,6.7,134.379259,22404.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,HP,Inter,None,21,256.9,11.9,213.571058,30268.096783,432.8,16.8,354.972381,47165.969524,175.9,4.9,141.401323,16897.872741
78,LP,Sporadic,10_RHA_gig,5438,170.6,16.2,263.509914,23464.388597,184.0,16.9,269.534051,25258.077023,13.4,0.7,6.024137,1793.688425
79,HP,Sporadic,00_non_Gig,3,40.0,3.0,26.630556,3611.333333,74.7,5.3,68.207778,7010.000000,34.7,2.3,41.577222,3398.666667
80,MP,Intra,10_RHA_gig,1934,215.1,18.4,313.331404,27550.354858,249.2,20.3,343.797608,32109.921633,34.1,1.9,30.466204,4559.566775


In [ ]:
query="""
 with appo as(
select captain_id,gig_segment,row_number() over(partition by captain_id order by run_date desc) as r
from reports_internal.captain_gig_appography_v4
where run_date='20250107'
),
byob_confirmed as (
    select distinct
      min(yyyymmdd) over(partition by eventprops__eventprops__userid order by yyyymmdd) as yyyymmdd,
      eventprops__eventprops__userid as captain_id
    from
      canonical.iceberg_production_data_raw_clevertap_captain_clevertap_captain_events_v2_eventname_byob_screen_immutable
    where eventprops__eventprops__action='plan_confirmed'
    and yyyymmdd between '20250129' and '20250214'
),
byob_confmd as(
   select a.*,b.performance_segment,b.consistency_segment,c.gig_segment
   from byob_confirmed a
   left join (select captain_id,performance_segment,consistency_segment
    from(
        select distinct captain_id,performance_segment,consistency_segment
        from  hive.experiments_internal.captain_byob_experiment_2_base_test 
    )) b
    on a.captain_id=b.captain_id 
    left join appo c
    on a.captain_id=c.captain_id 
),
byob_test_caps as(
  select a.yyyymmdd,performance_segment,consistency_segment,gig_segment,a.captain_id,date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d')) as cohot,
      yyyymmdd as post_start_date,
      date_format(date_add('day',date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d')),date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as post_end_date,
      date_format(date_add('day',-1,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_end_date,
      date_format(date_add('day',-(date_diff('day',date_parse(yyyymmdd,'%Y%m%d'),date_parse('20250224','%Y%m%d'))+1),date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_start_date
  from byob_confmd a
),
orders as(
 select yyyymmdd,captain_id,count(distinct order_id) as numOrders
 from orders.order_logs_fact
 where order_status='dropped'
 and yyyymmdd between '20241223' and '20250224'
 and captain_id in (select captain_id from byob_test_caps)
 group by 1,2
),
cm as (
  select yyyymmdd,captain_id,sum(total_login_hours) as login_hrs
  from datasets.captain_metrics as cm
  where yyyymmdd between '20241223' and '20250224'
  and captain_id in (select captain_id from byob_test_caps)
  group by 1,2
),
totalEarning as( select yyyymmdd,rider_id as captain_id,
    coalesce(sum(case when service = 'Link' then captain_earnings end),0) as earning
    from captain.captain_transactions_snapshot
            where 1=1
            and (transaction_type = 'orders' or transaction_type='incentives')
            and yyyymmdd between '20250118' and '20250219'
            and rider_id in (select captain_id from byob_test_caps)
    group by 1,2
),
metric_orders as (
   select a.yyyymmdd,a.captain_id,a.numOrders,b.login_hrs
   from orders a
   inner join cm b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
   where a.numOrders>0
),
metric_order_earning as(
  select a.yyyymmdd,a.captain_id,a.numOrders,a.login_hrs,b.earning
   from metric_orders a
   left join totalEarning b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
),
finalTbl as (
   select a.performance_segment,a.consistency_segment,a.gig_segment,
        a.captain_id,
        count(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.yyyymmdd end) as pre_netDays,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.numOrders else 0 end) as pre_numOrders,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.login_hrs else 0 end) as pre_login_hrs,
        sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.earning else 0 end) as pre_total_earnings,
        count(distinct case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.yyyymmdd end) as post_netDays,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.numOrders else 0  end) as post_numOrders,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.login_hrs else 0 end) as post_login_hrs,
        sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.earning else 0 end) as post_total_earnings
    from byob_test_caps a
    inner join metric_order_earning b
    on a.captain_id=b.captain_id
    group by 1,2,3,4,5
),
analysisTbl as (
    select performance_segment,yyyymmdd,consistency_segment,gig_segment,count(distinct captain_id) as totalCaptains,sum(pre_numOrders) as preOrders,sum(pre_netDays) as preNetDays,sum(pre_login_hrs) as preLoginHrs,
    sum(pre_total_earnings) as preTotalEarnings,
    sum(post_total_earnings) as postTotalEarnings,
    sum(post_numOrders) as postOrders,sum(post_netDays) as postNetDays,
    sum(post_login_hrs) as postLoginHrs
    from finalTbl
    group by 1,2,3,4
)
select * , (RPR_Post - RPR_Pre) as RPR_Change, (Net_days_Post - Net_days_Pre) as Net_days_Change, (LH_Post - LH_Pre) as LH_Change,(Earnings_post - Earnings_pre) as Earnings_change
from (
select yyyymmdd,performance_segment,consistency_segment,gig_segment,
totalCaptains as Total_Captains,
(preOrders*1.0/totalCaptains) as RPR_Pre,  (preNetDays*1.0/totalCaptains) as Net_days_Pre, (preLoginHrs*1.0 / totalCaptains) as LH_Pre,(preTotalEarnings*1.0/totalCaptains) as Earnings_pre,
(postOrders*1.0/totalCaptains) as RPR_Post,  (postNetDays*1.0/totalCaptains) as Net_days_Post, (postLoginHrs*1.0 / totalCaptains) as LH_Post,(postTotalEarnings*1.0/totalCaptains) as Earnings_post
from analysisTbl
)
"""
hypothesis=pd.read_sql(query,metabase_connection)

KeyboardInterrupt: 

In [44]:
hypothesis.to_clipboard()

In [56]:
autoPay

,Total_Captains,RPR_Pre,Net_days_Pre,LH_Pre,Earnings_pre,RPR_Post,Net_days_Post,LH_Post,Earnings_post,RPR_Change,Net_days_Change,LH_Change,Earnings_change
0,26303,79.7,15.9,130.41701,8.941228,59.8,11.5,91.414158,2.098357,-19.9,-4.4,-39.002853,-6.842872


In [60]:
autoPay

,Total_Captains,RPR_Pre,Net_days_Pre,LH_Pre,Earnings_pre,RPR_Post,Net_days_Post,LH_Post,Earnings_post,RPR_Change,Net_days_Change,LH_Change,Earnings_change
0,24701,87.3,17.5,143.58478,6.745868,91.8,17.4,138.395271,2.709548,4.5,-0.1,-5.189509,-4.036321


In [63]:
whenCreated=PXCautoPay

In [67]:
whenCreated.to_clipboard()

In [72]:
GXPXCautoPayyyymmdd=GXPXCautoPay

In [74]:
autoPay.to_clipboard()

In [75]:
GXPXCautoPayyyymmdd.to_clipboard()

In [82]:
GPCautoPay.to_clipboard()

In [98]:
q="""
with upsdate_date as (
     select userid as captain_id,
     updated_yyyymmdd as updatedate,
     yyyymmdd as createdon,
     yyyymmdd as post_start_date,
     date_format(date_add('day',28,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as post_end_date,
     date_format(date_add('day',-1,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_end_date,
     date_format(date_add('day',-29,date_parse(yyyymmdd,'%Y%m%d')),'%Y%m%d') as pre_start_date,
     row_number() over(partition by userid order by updated_yyyymmdd desc) as rank,status
     from canonical.iceberg_domain_payments_captain_autopay_immutable
     where updated_yyyymmdd between'20241001' and '20250126'
     
   ),active_mandate as (
     select *
     from upsdate_date a
     where a.rank=1
     and lower(a.status) like '%mandateactive%'
   ),
   pc as (
      select captain_id,
           case
               when lower(consistency_segment)='inter' then 'Intra'
               when lower(consistency_segment)='intra' then 'Inter'
               else consistency_segment
               end as consistency_segment,performance_segment
    from datasets_internal.captain_subservice_segments_v1 
    where run_date='20250225'
   ),
   appo as(
    select captain_id,gig_segment
    from reports_internal.captain_gig_appography_v4
    where run_date='20250107'
   ),
   am as (
     select a.*,b.consistency_segment,b.performance_segment,c.gig_segment
     from active_mandate a
     left join pc b
     on a.captain_id=b.captain_id
     left join appo c
     on a.captain_id=c.captain_id
   ),
   orders as(
    select yyyymmdd,captain_id,count(distinct order_id) as numOrders
    from orders.order_logs_fact
    where order_status='dropped'
    and yyyymmdd between '20240901' and '2025024'
    and captain_id in (select captain_id from active_mandate)
    group by 1,2
  ),
cm as (
  select yyyymmdd,captain_id,sum(total_login_hours) as login_hrs
  from datasets.captain_metrics as cm
  where yyyymmdd between '20240901' and '20250224'
  and captain_id in (select captain_id from active_mandate)
  group by 1,2
),
totalEarning as( select yyyymmdd,rider_id as captain_id,
    sum(captain_earnings) as earning
    from captain.captain_transactions_snapshot
            where 1=1
            and (transaction_type = 'orders' or transaction_type='incentives')
            and yyyymmdd between '20240901' and '20250224'
            and rider_id in (select captain_id from active_mandate)
    group by 1,2
),
metric_orders as (
   select a.yyyymmdd,a.captain_id,a.numOrders,b.login_hrs
   from orders a
   left join cm b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
   where a.numOrders>0
),
metric_order_earning as(
  select a.yyyymmdd,a.captain_id,a.numOrders,a.login_hrs,b.earning
   from metric_orders a
   left join totalEarning b
   on a.captain_id=b.captain_id
   and a.yyyymmdd=b.yyyymmdd
),
 finalTbl as (
    select
         a.captain_id,
         count(distinct case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.yyyymmdd end) as pre_netDays,
         sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.numOrders else 0 end) as pre_numOrders,
         sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.login_hrs else 0 end) as pre_login_hrs,
         sum(case when b.yyyymmdd between a.pre_start_date and a.pre_end_date then b.earning else 0 end) as pre_total_earnings,
         count(distinct  case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.yyyymmdd end) as post_netDays,
         sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.numOrders else 0 end) as post_numOrders,
         sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.login_hrs else 0 end) as post_login_hrs,
         sum(case when b.yyyymmdd between a.post_start_date and a.post_end_date then b.earning else 0 end) as post_total_earnings
     from am a
     inner join metric_order_earning b
     on a.captain_id=b.captain_id
     group by 1
 ),
 analysisTbl as (
     select  count(captain_id) as totalCaptains,sum(pre_numOrders) as preOrders,sum(pre_netDays) as preNetDays,sum(pre_login_hrs) as preLoginHrs,
     sum(pre_total_earnings) as preTotalEarnings,
     sum(post_total_earnings) as postTotalEarnings,
     sum(post_numOrders) as postOrders,sum(post_netDays) as postNetDays,
     sum(post_login_hrs) as postLoginHrs
     from finalTbl
 )
 select * , (RPR_Post - RPR_Pre) as RPR_Change, (Net_days_Post - Net_days_Pre) as Net_days_Change, (LH_Post - LH_Pre) as LH_Change,(Earnings_post - Earnings_pre) as Earnings_change
from (
select 
totalCaptains as Total_Captains,
(preOrders*1.0/totalCaptains) as RPR_Pre,  (preNetDays*1.0/totalCaptains) as Net_days_Pre, (preLoginHrs*1.0 / totalCaptains) as LH_Pre,(preTotalEarnings*1.0/totalCaptains) as Earnings_pre,
(postOrders*1.0/totalCaptains) as RPR_Post,  (postNetDays*1.0/totalCaptains) as Net_days_Post, (postLoginHrs*1.0 / totalCaptains) as LH_Post,(postTotalEarnings*1.0/totalCaptains) as Earnings_post
from analysisTbl
)
"""
autoPay=pd.read_sql(q,metabase_connection)

In [96]:
GPCautoPay.to_clipboard()

In [100]:
autoPay.to_clipboard()

In [102]:
q="""
  select yyyymmdd,count(*) as totalInitiation
        from clevertap.captain_bot_initiated_immutable
        where yyyymmdd between '20250206' and '20250224'
        and event_props_context='bot_context__bob'
        and event_props_event_name='BOT_INITIATED'
        group by 1
"""
precr=pd.read_sql(q,metabase_connection)

In [103]:
q="""
  select yyyymmdd,count(*) as totalInitiation
        from clevertap.captain_bot_initiated_immutable
        where yyyymmdd between '20250218' and '20250206'
        and event_props_context='bot_context__bob'
        and event_props_event_name='BOT_INITIATED'
        group by 1
"""
postcr=pd.read_sql(q,metabase_connection)

In [104]:
q=""" with bot_initiated as (
        select
            yyyymmdd, count(*) as totalInitiated
        from clevertap.captain_bot_initiated_immutable
        where yyyymmdd between '20250206' and '20250224'
        and event_props_context='bot_context__bob'
        and event_props_event_name='BOT_INITIATED'
        group by 1
    ), 
    rating_captains as (
        select yyyymmdd,count(*) as totalRatingGiven
        from clevertap.captain_bot_csat_immutable
        where event_props_context='bot_context__bob'
        and yyyymmdd between '20250206' and '20250224'
        group by 1
    ),
    call_data as (
       select yyyymmdd,count(*) as totalCalls
       from clevertap.captain_bot_call_support_immutable 
       where event_props_event_name='BOT_CALL_SUPPORT'
       and yyyymmdd between '20250206' and '20250224'
       and event_props_context='bot_context__bob'
       group by 1
    )
    select a.yyyymmdd,a.totalInitiated,b.totalRatingGiven,c.totalCalls
    from bot_initiated a
    left join rating_captains b
    on b.yyyymmdd=a.yyyymmdd
    left join call_data c
    on c.yyyymmdd=a.yyyymmdd"""

In [105]:
postP=pd.read_sql(q,metabase_connection)

In [109]:
q="""with bot_initiated as (
        select
            yyyymmdd, count(*) as totalInitiated
        from clevertap.captain_bot_initiated_immutable
        where yyyymmdd between '20250118' and '20250206'
        and event_props_event_name='BOT_INITIATED'
        group by 1
    ), 
    rating_captains as (
        select yyyymmdd,count(*) as totalRatingGiven
        from clevertap.captain_bot_csat_immutable
        where event_props_context='bot_context__bob'
        and yyyymmdd between '20250118' and '20250206'
        group by 1
    ),
    call_data as (
       select yyyymmdd,count(*) as totalCalls
       from clevertap.captain_bot_call_support_immutable 
       where event_props_event_name='BOT_CALL_SUPPORT'
       and yyyymmdd between '20250118' and '20250206'
       group by 1
    )
    select a.yyyymmdd,a.totalInitiated,b.totalRatingGiven,c.totalCalls
    from bot_initiated a
    left join rating_captains b
    on b.yyyymmdd=a.yyyymmdd
    left join call_data c
    on c.yyyymmdd=a.yyyymmdd"""
preP=pd.read_sql(q,metabase_connection)

In [111]:
preP.to_clipboard()